# Sampling Strategy Comparison

Compare different negative sampling strategies for the link prediction task.
- **Graph**: 2017–2024 co-occurrence graph (pre-built)
- **Test edges**: 2025 new edges between existing nodes
- **Budget**: 50,000 candidate pairs per strategy
- **Goal**: measure how many true positives each strategy captures

In [1]:
import pickle
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import networkx as nx

warnings.filterwarnings('ignore')

GRAPH_PATH = '../data/link_prediction/llm_concept_datasets/graph_2017_2024.pkl'
TEST_EDGES_PATH = '../data/link_prediction/llm_concept_datasets/test_edges_test_2025.pkl'
N_SAMPLE = 50_000
SEED = 42
rng = np.random.default_rng(SEED)

## Load data

In [2]:
with open(GRAPH_PATH, 'rb') as f:
    G = pickle.load(f)

with open(TEST_EDGES_PATH, 'rb') as f:
    raw_test_edges = pickle.load(f)

print(f'Graph — nodes: {G.number_of_nodes():,}  edges: {G.number_of_edges():,}')
print(f'Density: {nx.density(G):.6f}')
print(f'Raw test edges loaded: {len(raw_test_edges):,}')

Graph — nodes: 47,461  edges: 326,402
Density: 0.000290
Raw test edges loaded: 20,631


In [3]:
# Normalise: keep only edges where BOTH nodes exist in the train graph
node_set = set(G.nodes())
train_edge_set = {(min(u, v), max(u, v)) for u, v in G.edges()}

test_edges = set()
for u, v in raw_test_edges:
    if u in node_set and v in node_set:
        test_edges.add((min(u, v), max(u, v)))

n_nodes = G.number_of_nodes()
total_possible = n_nodes * (n_nodes - 1) // 2
total_non_edges = total_possible - G.number_of_edges()

print(f'Test positives (both nodes in graph): {len(test_edges):,}')
print(f'Total possible non-edges:             {total_non_edges:,}')
print(f'Baseline positive rate (random):      {len(test_edges)/total_non_edges:.6%}')
print(f'  ≈ 1 positive per {total_non_edges/max(len(test_edges),1):,.0f} random samples')

Test positives (both nodes in graph): 20,631
Total possible non-edges:             1,125,923,128
Baseline positive rate (random):      0.001832%
  ≈ 1 positive per 54,574 random samples


## Pre-compute graph structures

In [4]:
nodes = list(G.nodes())
nodes_arr = np.array(nodes)
neighbors = {n: set(G.neighbors(n)) for n in nodes}
degree = dict(G.degree())
strength = dict(G.degree(weight='weight'))

# Degree-proportional sampling probabilities
deg_arr = np.array([degree[n] for n in nodes], dtype=np.float64)
deg_probs = deg_arr / deg_arr.sum()

# Strength-proportional sampling probabilities
str_arr = np.array([strength[n] for n in nodes], dtype=np.float64)
str_probs = str_arr / str_arr.sum()

print('Neighbourhood structures ready.')
print(f'Avg degree: {deg_arr.mean():.1f}  Max degree: {int(deg_arr.max())}')

Neighbourhood structures ready.
Avg degree: 13.8  Max degree: 2444


## Sampling strategies

| Strategy | Description |
|---|---|
| **Random** | Uniform random non-edges — true-world baseline |
| **2-hop** | Non-edges sharing ≥1 common neighbour, weighted by common-neighbour count |
| **Resource Allocation (RA)** | Top non-edges by RA score = Σ 1/deg(z) over common neighbours |
| **Preferential Attachment (PA)** | Sample ∝ degree(u) × degree(v) |
| **Strength-biased** | Sample ∝ weight-strength(u) × weight-strength(v) |

In [6]:
def is_non_edge(u, v):
    pair = (min(u, v), max(u, v))
    return pair not in train_edge_set and u != v


def sample_random(n):
    candidates = set()
    while len(candidates) < n:
        idx = rng.integers(0, len(nodes_arr), size=(n * 4, 2))
        for i, j in idx:
            if i == j:
                continue
            u, v = nodes_arr[i], nodes_arr[j]
            pair = (min(u, v), max(u, v))
            if pair not in train_edge_set:
                candidates.add(pair)
            if len(candidates) >= n:
                break
    return list(candidates)[:n]


def sample_2hop(n, node_budget=5000):
    # Focus on higher-degree nodes to keep runtime manageable
    idx = rng.choice(len(nodes_arr), size=min(node_budget, len(nodes_arr)),
                     replace=False, p=deg_probs)
    sample_nodes = [nodes_arr[i] for i in idx]

    candidates = []
    weights = []
    seen = set()

    for node in sample_nodes:
        for nb in neighbors[node]:
            for target in neighbors[nb]:
                if target == node:
                    continue
                pair = (min(node, target), max(node, target))
                if pair in seen or pair in train_edge_set:
                    continue
                seen.add(pair)
                cn = len(neighbors[node] & neighbors[target])
                candidates.append(pair)
                weights.append(cn)

    print(f'  2-hop: {len(candidates):,} candidate pairs found')
    w = np.array(weights, dtype=float)
    w /= w.sum()
    n = min(n, len(candidates))
    idx = rng.choice(len(candidates), size=n, replace=False, p=w)
    return [candidates[i] for i in idx]


def sample_resource_allocation(n, node_budget=3000):
    idx = rng.choice(len(nodes_arr), size=min(node_budget, len(nodes_arr)),
                     replace=False, p=deg_probs)
    sample_nodes = [nodes_arr[i] for i in idx]

    scored = []
    seen = set()

    for node in sample_nodes:
        for nb in neighbors[node]:
            for target in neighbors[nb]:
                if target == node:
                    continue
                pair = (min(node, target), max(node, target))
                if pair in seen or pair in train_edge_set:
                    continue
                seen.add(pair)
                common = neighbors[node] & neighbors[target]
                ra = sum(1.0 / degree[z] for z in common if degree[z] > 0)
                if ra > 0:
                    scored.append((pair, ra))

    print(f'  RA: {len(scored):,} pairs with RA > 0')
    scored.sort(key=lambda x: x[1], reverse=True)
    return [pair for pair, _ in scored[:n]]


def sample_preferential_attachment(n):
    candidates = set()
    while len(candidates) < n:
        idx = rng.choice(len(nodes_arr), size=(n * 5, 2), p=deg_probs)
        for i, j in idx:
            if i == j:
                continue
            u, v = nodes_arr[i], nodes_arr[j]
            pair = (min(u, v), max(u, v))
            if pair not in train_edge_set:
                candidates.add(pair)
            if len(candidates) >= n:
                break
    return list(candidates)[:n]


def sample_strength_biased(n):
    candidates = set()
    while len(candidates) < n:
        idx = rng.choice(len(nodes_arr), size=(n * 5, 2), p=str_probs)
        for i, j in idx:
            if i == j:
                continue
            u, v = nodes_arr[i], nodes_arr[j]
            pair = (min(u, v), max(u, v))
            if pair not in train_edge_set:
                candidates.add(pair)
            if len(candidates) >= n:
                break
    return list(candidates)[:n]

## Run all strategies and count positives

In [7]:
strategies = {
    'Random': sample_random,
    '2-hop': sample_2hop,
    'Resource Allocation': sample_resource_allocation,
    'Preferential Attachment': sample_preferential_attachment,
    'Strength-biased': sample_strength_biased,
}

results = {}

for name, fn in strategies.items():
    print(f'\n{name}')
    pairs = fn(N_SAMPLE)
    hits  = sum(1 for p in pairs if p in test_edges)
    results[name] = {'pairs': pairs, 'n_sampled': len(pairs), 'n_positives': hits}
    print(f'    sampled {len(pairs):,}  positives {hits}')


Random
    sampled 50,000  positives 1

2-hop
  2-hop: 10,325,647 candidate pairs found
    sampled 50,000  positives 138

Resource Allocation
  RA: 7,341,970 pairs with RA > 0
    sampled 50,000  positives 1169

Preferential Attachment
    sampled 50,000  positives 84

Strength-biased
    sampled 50,000  positives 97


## Summary table

In [8]:
random_rate = results['Random']['n_positives'] / max(results['Random']['n_sampled'], 1)

rows = []
for name, r in results.items():
    n = r['n_sampled']
    hits = r['n_positives']
    rate = hits / max(n, 1)
    enrichment = rate / random_rate if random_rate > 0 else float('nan')
    coverage = hits / max(len(test_edges), 1)
    rows.append({
        'Strategy': name,
        'Sampled': n,
        'Positives': hits,
        'Positive rate': rate,
        'Enrichment (×)': enrichment,
        'Coverage of test': coverage,
    })

df = pd.DataFrame(rows)

# Formatting for display
df_display = df.copy()
df_display['Sampled'] = df_display['Sampled'].map('{:,}'.format)
df_display['Positives'] = df_display['Positives'].map('{:,}'.format)
df_display['Positive rate'] = df_display['Positive rate'].map('{:.4%}'.format)
df_display['Enrichment (×)'] = df_display['Enrichment (×)'].map(lambda x: f'{x:.1f}×' if not np.isnan(x) else 'baseline')
df_display['Coverage of test'] = df_display['Coverage of test'].map('{:.2%}'.format)

df_display.set_index('Strategy', inplace=True)
df_display

,Sampled,Positives,Positive rate,Enrichment (×),Coverage of test
Strategy,,,,,
Random,"50,000",1,0.0020%,1.0×,0.00%
2-hop,"50,000",138,0.2760%,138.0×,0.67%
Resource Allocation,"50,000","1,169",2.3380%,1169.0×,5.67%
Preferential Attachment,"50,000",84,0.1680%,84.0×,0.41%
Strength-biased,"50,000",97,0.1940%,97.0×,0.47%


# Sampling Strategy Analysis

**Random Sampling:** With only **1 true positive out of 50,000 samples**, random sampling confirms the extreme sparsity of the problem — a base rate of ~0.002%.

**Resource Allocation** achieves **1,169× enrichment**, finding **~8.5× more true positives** than the next best strategy (2-hop at 138). Roughly **1 in every 43 candidates** is a genuine future edge.

All structural strategies vastly outperform random, confirming that topology carries predictive signal, but with notable spread:

- **2-hop (138×)** — restricts to pairs sharing a neighbor but makes no distinction among them
- **Strength-biased (97×)** — modest improvement over Preferential Attachment via edge weights, still lags behind
- **Preferential Attachment (84×)** — weakest structural strategy; favoring hub-hub pairs over-predicts connections that never materialize ("popularity ≠ affinity")
